In [22]:
from pathlib import Path

import pandas as pd
import numpy as np
import json
import re
import subprocess
import tempfile
import hashlib

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_colwidth", 300)

PROJECT_DIR = Path("..").resolve()

PUBLIC_RAW = (
    PROJECT_DIR
    / "data"
    / "public_v2"
    / "raw"
)

REPORTS_DIR = (
    PROJECT_DIR
    / "reports"
    / "tables"
)

CATCHING_RESULTS = (
    PUBLIC_RAW
    / "04_catching_ux_flaws"
    / "results"
)

REPORTS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

In [23]:
candidates = pd.read_csv(
    REPORTS_DIR
    / "catching_source_candidates.csv"
)

recovery_review = pd.read_csv(
    REPORTS_DIR
    / "catching_source_recovery_review.csv"
)

print(
    "Candidates:",
    candidates.shape
)

print(
    "Recovery review:",
    recovery_review.shape
)

Candidates: (268, 17)
Recovery review: (30, 10)


In [24]:
exact_candidates = candidates[
    candidates[
        "exact_name_match"
    ] == True
].copy()

print(
    "Repositorios candidatos exactos:",
    len(exact_candidates)
)

print(
    "Sitios:",
    exact_candidates[
        "site"
    ].nunique()
)

Repositorios candidatos exactos: 73
Sitios: 28


In [25]:
def flatten_text(value):

    if value is None:
        return None

    if isinstance(
        value,
        str
    ):
        return value

    if isinstance(
        value,
        list
    ):
        return "\n".join(
            flatten_text(x)
            or ""
            for x in value
        )

    if isinstance(
        value,
        dict
    ):
        return "\n".join(
            flatten_text(x)
            or ""
            for x in value.values()
        )

    return str(value)

In [26]:
evidence_records = []

for path in sorted(
    CATCHING_RESULTS.rglob(
        "*.json"
    )
):

    site = path.parent.name

    evaluation = (
        path.stem
        .lower()
        .replace(" ", "")
    )

    with open(
        path,
        "r",
        encoding="utf-8"
    ) as f:

        data = json.load(f)

    for row_number, item in enumerate(
        data
    ):

        snippet = flatten_text(
            item.get(
                "CodeSnippet"
            )
        )

        reference = flatten_text(
            item.get(
                "CodeReference"
            )
        )

        if not snippet:
            continue

        evidence_records.append({
            "site":
                site,

            "evaluation":
                evaluation,

            "row_number":
                row_number,

            "heuristic":
                item.get(
                    "Heuristic"
                ),

            "code_reference":
                reference,

            "code_snippet":
                snippet
        })

catching_evidence = pd.DataFrame(
    evidence_records
)

print(
    catching_evidence.shape
)

display(
    catching_evidence.head(20)
)

(895, 6)


,site,evaluation,row_number,heuristic,code_reference,code_snippet
0,Business-Website,eval1,0,Visibility of system status,index.html (lines involving .get-started),"<div class=""get-started"">\n <button class=""get-started-login button-hover"">Login</button>\n <button class=""get-started-free button-hover"">Start for Free</button>\n</div>"
1,Business-Website,eval1,1,Match between system and the real world,index.html (header and buttons),"<span class=""logo"">Business Homepage</span>"
2,Business-Website,eval1,2,User control and freedom,index.html (global navigation and footer absent),<!-- No <nav> or <footer> present -->
3,Business-Website,eval1,3,Consistency and standards,"style.css (button styles), index.html (buttons)","<button class=""get-started-login button-hover"">Login</button>"
4,Business-Website,eval1,4,Error prevention,index.html,<!-- No forms with validation found -->
5,Business-Website,eval1,5,Recognition rather than recall,index.html (header and buttons),"<span class=""hiring-text"">We're hiring</span>"
6,Business-Website,eval1,6,Flexibility and efficiency of use,index.html (global layout),"<body>\n <div class=""fluid-container"">"
7,Business-Website,eval1,7,Aesthetic and minimalist design,"style.css (.blue-back, .title-task-text)",.blue-back {\n background-color: #0083ee;\n width: 100%;\n height: 575px;\n}
8,Business-Website,eval1,8,"Help users recognize, diagnose, and recover from errors",index.html,<!-- No error-handling components present -->
9,Business-Website,eval1,9,Help and documentation,index.html (no help section or links present),<!-- Help/documentation section not included -->


In [27]:
def clean_code_snippet(text):

    if pd.isna(text):
        return ""

    text = str(text)

    # Quitar fences Markdown
    text = re.sub(
        r"```[a-zA-Z0-9_-]*",
        "",
        text
    )

    text = text.replace(
        "```",
        ""
    )

    return text.strip()

In [28]:
def compact_code(text):

    text = clean_code_snippet(
        text
    )

    return re.sub(
        r"\s+",
        "",
        text
    )

In [29]:
catching_evidence[
    "snippet_compact"
] = catching_evidence[
    "code_snippet"
].apply(
    compact_code
)

catching_evidence[
    "snippet_length"
] = catching_evidence[
    "snippet_compact"
].str.len()

In [30]:
validation_snippets = (
    catching_evidence[
        catching_evidence[
            "snippet_length"
        ] >= 80
    ]
    .copy()
)

print(
    "Snippets utilizables:",
    len(validation_snippets)
)

Snippets utilizables: 154


In [31]:
SOURCE_EXTENSIONS = {
    ".html",
    ".htm",
    ".css",
    ".js",
    ".jsx",
    ".ts",
    ".tsx",
    ".vue",
    ".php",
    ".md"
}

SKIP_DIRS = {
    ".git",
    "node_modules",
    "vendor",
    "dist",
    "build",
    ".next",
    "coverage"
}

In [32]:
def iter_source_files(
    repo_dir
):

    for path in repo_dir.rglob(
        "*"
    ):

        if not path.is_file():
            continue

        if any(
            part in SKIP_DIRS
            for part in path.parts
        ):
            continue

        if (
            path.suffix.lower()
            not in SOURCE_EXTENSIONS
        ):
            continue

        # Evitar archivos enormes
        try:

            if (
                path.stat().st_size
                >
                2_000_000
            ):
                continue

        except OSError:
            continue

        yield path

In [33]:
def validate_repository(
    site,
    repo_full_name,
    clone_url,
    snippets
):

    result = {
        "site":
            site,

        "repo_full_name":
            repo_full_name,

        "clone_ok":
            False,

        "source_files":
            0,

        "snippets_tested":
            0,

        "matched_snippets":
            0,

        "matched_evaluations":
            0,

        "matched_files":
            None,

        "error":
            None
    }

    with tempfile.TemporaryDirectory() as tmp:

        repo_dir = (
            Path(tmp)
            / "repo"
        )

        try:

            process = subprocess.run(
                [
                    "git",
                    "clone",
                    "--depth",
                    "1",
                    "--quiet",
                    clone_url,
                    str(repo_dir)
                ],
                capture_output=True,
                text=True,
                timeout=180
            )

        except Exception as e:

            result[
                "error"
            ] = str(e)

            return result

        if process.returncode != 0:

            result[
                "error"
            ] = (
                process.stderr[
                    :500
                ]
            )

            return result

        result[
            "clone_ok"
        ] = True

        source_files = list(
            iter_source_files(
                repo_dir
            )
        )

        result[
            "source_files"
        ] = len(
            source_files
        )

        # Leer una sola vez
        repo_contents = []

        for path in source_files:

            try:

                text = path.read_text(
                    encoding="utf-8",
                    errors="ignore"
                )

            except Exception:
                continue

            repo_contents.append(
                (
                    path,
                    compact_code(
                        text
                    )
                )
            )

        matched_hashes = set()
        matched_evals = set()
        matched_files = set()

        for _, row in snippets.iterrows():

            snippet = row[
                "snippet_compact"
            ]

            if (
                not snippet
                or
                len(snippet) < 80
            ):
                continue

            result[
                "snippets_tested"
            ] += 1

            snippet_hash = (
                hashlib.sha256(
                    snippet.encode(
                        "utf-8"
                    )
                )
                .hexdigest()
            )

            for path, content in repo_contents:

                if snippet in content:

                    matched_hashes.add(
                        snippet_hash
                    )

                    matched_evals.add(
                        row[
                            "evaluation"
                        ]
                    )

                    matched_files.add(
                        str(
                            path.relative_to(
                                repo_dir
                            )
                        )
                    )

                    break

        result[
            "matched_snippets"
        ] = len(
            matched_hashes
        )

        result[
            "matched_evaluations"
        ] = len(
            matched_evals
        )

        result[
            "matched_files"
        ] = " | ".join(
            sorted(
                matched_files
            )
        )

    return result

In [34]:
test_candidate = exact_candidates[
    exact_candidates[
        "site"
    ]
    ==
    "academic-website-template"
].iloc[0]

test_snippets = validation_snippets[
    validation_snippets[
        "site"
    ]
    ==
    test_candidate[
        "site"
    ]
]

In [35]:
test_result = validate_repository(

    site=
        test_candidate[
            "site"
        ],

    repo_full_name=
        test_candidate[
            "repo_full_name"
        ],

    clone_url=
        test_candidate[
            "clone_url"
        ],

    snippets=
        test_snippets
)

test_result

{'site': 'academic-website-template',
 'repo_full_name': 'sbryngelson/academic-website-template',
 'clone_ok': True,
 'source_files': 31,
 'snippets_tested': 8,
 'matched_snippets': 0,
 'matched_evaluations': 0,
 'matched_files': '',
 'error': None}

In [36]:
single_sites = set(
    recovery_review.loc[
        recovery_review[
            "review_status"
        ]
        ==
        "REVIEW_SINGLE_CANDIDATE",
        "site"
    ]
)

single_candidates = (
    exact_candidates[
        exact_candidates[
            "site"
        ].isin(
            single_sites
        )
    ]
    .copy()
)

print(
    single_candidates.shape
)

(11, 17)


In [37]:
single_validation_records = []

for i, row in single_candidates.iterrows():

    site = row[
        "site"
    ]

    print(
        "Validando:",
        site,
        "->",
        row[
            "repo_full_name"
        ]
    )

    snippets = validation_snippets[
        validation_snippets[
            "site"
        ]
        ==
        site
    ]

    result = validate_repository(
        site=
            site,

        repo_full_name=
            row[
                "repo_full_name"
            ],

        clone_url=
            row[
                "clone_url"
            ],

        snippets=
            snippets
    )

    single_validation_records.append(
        result
    )

Validando: academic-website-template -> sbryngelson/academic-website-template
Validando: conference-website-template -> mikepierce/conference-website-template
Validando: jekyll-inclusion -> website-templates/jekyll-inclusion
Validando: marketing-agency_one-page-template -> website-templates/marketing-agency_one-page-template
Validando: material-dashboard -> laravel-frontend-presets/material-dashboard
Validando: portfolio_one-page-template -> website-templates/portfolio_one-page-template
Validando: react-saas-template -> dunky11/react-saas-template
Validando: simple-product_responsive-template -> website-templates/simple-product_responsive-template
Validando: startbootstrap-3-col-portfolio -> StartBootstrap/startbootstrap-3-col-portfolio
Validando: startbootstrap-simple-sidebar -> StartBootstrap/startbootstrap-simple-sidebar
Validando: travel_fluid-template -> website-templates/travel_fluid-template


In [38]:
single_validation = pd.DataFrame(
    single_validation_records
)

display(
    single_validation
)

,site,repo_full_name,clone_ok,source_files,snippets_tested,matched_snippets,matched_evaluations,matched_files,error
0,academic-website-template,sbryngelson/academic-website-template,True,31,8,0,0,,None
1,conference-website-template,mikepierce/conference-website-template,True,8,2,2,1,index.html,None
2,jekyll-inclusion,website-templates/jekyll-inclusion,True,47,7,0,0,,None
3,marketing-agency_one-page-template,website-templates/marketing-agency_one-page-template,True,34,1,0,0,,None
4,material-dashboard,laravel-frontend-presets/material-dashboard,True,156,8,0,0,,None
5,portfolio_one-page-template,website-templates/portfolio_one-page-template,True,34,4,0,0,,None
6,react-saas-template,dunky11/react-saas-template,True,99,2,1,1,src/logged_in/components/dashboard/AccountInformationArea.js,None
7,simple-product_responsive-template,website-templates/simple-product_responsive-template,True,21,5,0,0,,None
8,startbootstrap-3-col-portfolio,StartBootstrap/startbootstrap-3-col-portfolio,True,4,12,10,2,index.html,None
9,startbootstrap-simple-sidebar,StartBootstrap/startbootstrap-simple-sidebar,True,14,11,1,1,src/js/scripts.js,None


In [39]:
def classify_validation(
    row
):

    if not row[
        "clone_ok"
    ]:
        return "CLONE_ERROR"

    if (
        row[
            "matched_snippets"
        ] >= 2
        and
        row[
            "matched_evaluations"
        ] >= 2
    ):
        return "STRONG_MATCH"

    if (
        row[
            "matched_snippets"
        ] >= 1
    ):
        return "MODERATE_MATCH"

    return "NO_MATCH"

In [40]:
single_validation[
    "validation_status"
] = single_validation.apply(
    classify_validation,
    axis=1
)

print(
    single_validation[
        "validation_status"
    ].value_counts(
        dropna=False
    )
)

display(
    single_validation
)

validation_status
NO_MATCH          7
MODERATE_MATCH    3
STRONG_MATCH      1
Name: count, dtype: int64


,site,repo_full_name,clone_ok,source_files,snippets_tested,matched_snippets,matched_evaluations,matched_files,error,validation_status
0,academic-website-template,sbryngelson/academic-website-template,True,31,8,0,0,,None,NO_MATCH
1,conference-website-template,mikepierce/conference-website-template,True,8,2,2,1,index.html,None,MODERATE_MATCH
2,jekyll-inclusion,website-templates/jekyll-inclusion,True,47,7,0,0,,None,NO_MATCH
3,marketing-agency_one-page-template,website-templates/marketing-agency_one-page-template,True,34,1,0,0,,None,NO_MATCH
4,material-dashboard,laravel-frontend-presets/material-dashboard,True,156,8,0,0,,None,NO_MATCH
5,portfolio_one-page-template,website-templates/portfolio_one-page-template,True,34,4,0,0,,None,NO_MATCH
6,react-saas-template,dunky11/react-saas-template,True,99,2,1,1,src/logged_in/components/dashboard/AccountInformationArea.js,None,MODERATE_MATCH
7,simple-product_responsive-template,website-templates/simple-product_responsive-template,True,21,5,0,0,,None,NO_MATCH
8,startbootstrap-3-col-portfolio,StartBootstrap/startbootstrap-3-col-portfolio,True,4,12,10,2,index.html,None,STRONG_MATCH
9,startbootstrap-simple-sidebar,StartBootstrap/startbootstrap-simple-sidebar,True,14,11,1,1,src/js/scripts.js,None,MODERATE_MATCH


In [41]:
single_validation.to_csv(
    REPORTS_DIR
    / "catching_single_candidate_validation.csv",
    index=False
)

In [42]:
print(
    "catching_single_candidate_validation.csv",
    "✅"
    if (
        REPORTS_DIR
        / "catching_single_candidate_validation.csv"
    ).exists()
    else "❌"
)

catching_single_candidate_validation.csv ✅


In [43]:
multiple_sites = set(
    recovery_review.loc[
        recovery_review[
            "review_status"
        ]
        ==
        "REVIEW_MULTIPLE_CANDIDATES",
        "site"
    ]
)

print(
    "Sitios con múltiples candidatos:",
    len(multiple_sites)
)

Sitios con múltiples candidatos: 17


In [44]:
multiple_candidates = (
    exact_candidates[
        exact_candidates[
            "site"
        ].isin(
            multiple_sites
        )
    ]
    .drop_duplicates(
        subset=[
            "site",
            "repo_full_name"
        ]
    )
    .copy()
)

print(
    "Sitios:",
    multiple_candidates[
        "site"
    ].nunique()
)

print(
    "Repositorios a validar:",
    len(multiple_candidates)
)

Sitios: 17
Repositorios a validar: 62


In [45]:
candidate_distribution = (
    multiple_candidates
    .groupby("site")
    .size()
    .sort_values(
        ascending=False
    )
)

display(
    candidate_distribution
)

site
Electronics-eCommerce-Shop-With-Admin-Dashboard-NextJS-NodeJS-main    9
startbootstrap-one-page-wonder                                        8
Discord-bot-website-template                                          6
beginner-html-site-styled                                             6
startbootstrap-landing-page                                           4
academicpages.github.io                                               3
charity_responsive-template                                           3
coreui-free-react-admin-template                                      3
simple-gallery_responsive-template                                    3
startbootstrap-creative                                               3
startbootstrap-freelancer                                             2
startbootstrap-resume                                                 2
personal-website-template                                             2
single-page-small-business-web-application                 

In [46]:
multiple_validation_records = []

total = len(
    multiple_candidates
)

for position, (_, row) in enumerate(
    multiple_candidates.iterrows(),
    start=1
):

    site = row[
        "site"
    ]

    repo = row[
        "repo_full_name"
    ]

    print(
        f"[{position}/{total}]",
        site,
        "->",
        repo
    )

    snippets = validation_snippets[
        validation_snippets[
            "site"
        ]
        ==
        site
    ]

    result = validate_repository(
        site=
            site,

        repo_full_name=
            repo,

        clone_url=
            row[
                "clone_url"
            ],

        snippets=
            snippets
    )

    multiple_validation_records.append(
        result
    )

[1/62] Discord-bot-website-template -> Neovenchy/Discord-bot-website-template
[2/62] Discord-bot-website-template -> hadi-4100/Discord-bot-website-template
[3/62] Discord-bot-website-template -> preetsuthar17/discord-bot-website-template
[4/62] Discord-bot-website-template -> kevinjosethomas/discord-bot-website-template
[5/62] Discord-bot-website-template -> RedolentHalo/discord-bot-website-template
[6/62] Discord-bot-website-template -> DiscordWasMyIdea/Discord-bot-website-template
[7/62] Electronics-eCommerce-Shop-With-Admin-Dashboard-NextJS-NodeJS-main -> Rajsu12/Electronics-eCommerce-Shop-With-Admin-Dashboard-NextJS-NodeJS-main
[8/62] Electronics-eCommerce-Shop-With-Admin-Dashboard-NextJS-NodeJS-main -> prin2603/Electronics-eCommerce-Shop-With-Admin-Dashboard-NextJS-NodeJS-main
[9/62] Electronics-eCommerce-Shop-With-Admin-Dashboard-NextJS-NodeJS-main -> jersonjb/Electronics-eCommerce-Shop-With-Admin-Dashboard-NextJS-NodeJS-main
[10/62] Electronics-eCommerce-Shop-With-Admin-Dashboar

In [47]:
multiple_validation = pd.DataFrame(
    multiple_validation_records
)

print(
    multiple_validation.shape
)

display(
    multiple_validation.head(30)
)

(62, 9)


,site,repo_full_name,clone_ok,source_files,snippets_tested,matched_snippets,matched_evaluations,matched_files,error
0,Discord-bot-website-template,Neovenchy/Discord-bot-website-template,True,6,5,4,1,src/index.html,None
1,Discord-bot-website-template,hadi-4100/Discord-bot-website-template,True,4,5,0,0,,None
2,Discord-bot-website-template,preetsuthar17/discord-bot-website-template,True,12,5,0,0,,None
3,Discord-bot-website-template,kevinjosethomas/discord-bot-website-template,True,19,5,0,0,,None
4,Discord-bot-website-template,RedolentHalo/discord-bot-website-template,True,4,5,0,0,,None
5,Discord-bot-website-template,DiscordWasMyIdea/Discord-bot-website-template,True,5,5,0,0,,None
6,Electronics-eCommerce-Shop-With-Admin-Dashboard-NextJS-NodeJS-main,Rajsu12/Electronics-eCommerce-Shop-With-Admin-Dashboard-NextJS-NodeJS-main,True,5,0,0,0,,None
7,Electronics-eCommerce-Shop-With-Admin-Dashboard-NextJS-NodeJS-main,prin2603/Electronics-eCommerce-Shop-With-Admin-Dashboard-NextJS-NodeJS-main,True,117,0,0,0,,None
8,Electronics-eCommerce-Shop-With-Admin-Dashboard-NextJS-NodeJS-main,jersonjb/Electronics-eCommerce-Shop-With-Admin-Dashboard-NextJS-NodeJS-main,True,117,0,0,0,,None
9,Electronics-eCommerce-Shop-With-Admin-Dashboard-NextJS-NodeJS-main,bhaskarbrsolutions/Electronics-eCommerce-Shop-With-Admin-Dashboard-NextJS-NodeJS-main,True,117,0,0,0,,None


In [48]:
def classify_repository_validation(
    row
):

    if not row[
        "clone_ok"
    ]:
        return "CLONE_ERROR"

    if (
        row[
            "snippets_tested"
        ] == 0
    ):
        return "NO_TESTABLE_EVIDENCE"

    if (
        row[
            "matched_snippets"
        ] >= 2
        and
        row[
            "matched_evaluations"
        ] >= 2
    ):
        return "STRONG_MATCH"

    if (
        row[
            "matched_snippets"
        ] >= 1
    ):
        return "MODERATE_MATCH"

    return "NO_MATCH"

In [49]:
multiple_validation[
    "validation_status"
] = multiple_validation.apply(
    classify_repository_validation,
    axis=1
)

print(
    multiple_validation[
        "validation_status"
    ].value_counts(
        dropna=False
    )
)

validation_status
NO_MATCH                34
NO_TESTABLE_EVIDENCE    19
MODERATE_MATCH           8
STRONG_MATCH             1
Name: count, dtype: int64


In [50]:
display(
    multiple_validation[
        [
            "site",
            "repo_full_name",
            "source_files",
            "snippets_tested",
            "matched_snippets",
            "matched_evaluations",
            "validation_status"
        ]
    ]
    .sort_values(
        [
            "site",
            "matched_evaluations",
            "matched_snippets"
        ],
        ascending=[
            True,
            False,
            False
        ]
    )
)

,site,repo_full_name,source_files,snippets_tested,matched_snippets,matched_evaluations,validation_status
0,Discord-bot-website-template,Neovenchy/Discord-bot-website-template,6,5,4,1,MODERATE_MATCH
1,Discord-bot-website-template,hadi-4100/Discord-bot-website-template,4,5,0,0,NO_MATCH
2,Discord-bot-website-template,preetsuthar17/discord-bot-website-template,12,5,0,0,NO_MATCH
3,Discord-bot-website-template,kevinjosethomas/discord-bot-website-template,19,5,0,0,NO_MATCH
4,Discord-bot-website-template,RedolentHalo/discord-bot-website-template,4,5,0,0,NO_MATCH
5,Discord-bot-website-template,DiscordWasMyIdea/Discord-bot-website-template,5,5,0,0,NO_MATCH
6,Electronics-eCommerce-Shop-With-Admin-Dashboard-NextJS-NodeJS-main,Rajsu12/Electronics-eCommerce-Shop-With-Admin-Dashboard-NextJS-NodeJS-main,5,0,0,0,NO_TESTABLE_EVIDENCE
7,Electronics-eCommerce-Shop-With-Admin-Dashboard-NextJS-NodeJS-main,prin2603/Electronics-eCommerce-Shop-With-Admin-Dashboard-NextJS-NodeJS-main,117,0,0,0,NO_TESTABLE_EVIDENCE
8,Electronics-eCommerce-Shop-With-Admin-Dashboard-NextJS-NodeJS-main,jersonjb/Electronics-eCommerce-Shop-With-Admin-Dashboard-NextJS-NodeJS-main,117,0,0,0,NO_TESTABLE_EVIDENCE
9,Electronics-eCommerce-Shop-With-Admin-Dashboard-NextJS-NodeJS-main,bhaskarbrsolutions/Electronics-eCommerce-Shop-With-Admin-Dashboard-NextJS-NodeJS-main,117,0,0,0,NO_TESTABLE_EVIDENCE


In [51]:
def decide_site_repository(
    group
):

    strong = group[
        group[
            "validation_status"
        ]
        ==
        "STRONG_MATCH"
    ]

    moderate = group[
        group[
            "validation_status"
        ]
        ==
        "MODERATE_MATCH"
    ]

    if len(strong) == 1:

        row = strong.iloc[0]

        return pd.Series({
            "repository_status":
                "CONFIRMED_STRONG",

            "selected_repo":
                row[
                    "repo_full_name"
                ],

            "matched_snippets":
                row[
                    "matched_snippets"
                ],

            "matched_evaluations":
                row[
                    "matched_evaluations"
                ]
        })

    if len(strong) > 1:

        return pd.Series({
            "repository_status":
                "AMBIGUOUS_MULTIPLE_STRONG",

            "selected_repo":
                None,

            "matched_snippets":
                np.nan,

            "matched_evaluations":
                np.nan
        })

    if len(moderate) == 1:

        row = moderate.iloc[0]

        return pd.Series({
            "repository_status":
                "PROVISIONAL_MODERATE",

            "selected_repo":
                row[
                    "repo_full_name"
                ],

            "matched_snippets":
                row[
                    "matched_snippets"
                ],

            "matched_evaluations":
                row[
                    "matched_evaluations"
                ]
        })

    if len(moderate) > 1:

        return pd.Series({
            "repository_status":
                "AMBIGUOUS_MULTIPLE_MODERATE",

            "selected_repo":
                None,

            "matched_snippets":
                np.nan,

            "matched_evaluations":
                np.nan
        })

    return pd.Series({
        "repository_status":
            "UNRESOLVED",

        "selected_repo":
            None,

        "matched_snippets":
            0,

        "matched_evaluations":
            0
    })

In [52]:
multiple_site_decisions = (
    multiple_validation
    .groupby(
        "site",
        group_keys=False
    )
    .apply(
        decide_site_repository,
        include_groups=False
    )
    .reset_index()
)

display(
    multiple_site_decisions
)

,site,repository_status,selected_repo,matched_snippets,matched_evaluations
0,Discord-bot-website-template,PROVISIONAL_MODERATE,Neovenchy/Discord-bot-website-template,4.0,1.0
1,Electronics-eCommerce-Shop-With-Admin-Dashboard-NextJS-NodeJS-main,UNRESOLVED,NaN,0.0,0.0
2,academicpages.github.io,UNRESOLVED,NaN,0.0,0.0
3,airspace-bootstrap,UNRESOLVED,NaN,0.0,0.0
4,beginner-html-site-styled,AMBIGUOUS_MULTIPLE_MODERATE,NaN,NaN,NaN
5,charity_responsive-template,UNRESOLVED,NaN,0.0,0.0
6,coreui-free-react-admin-template,AMBIGUOUS_MULTIPLE_MODERATE,NaN,NaN,NaN
7,design-studio_one-page-template,UNRESOLVED,NaN,0.0,0.0
8,personal-website-template,PROVISIONAL_MODERATE,The-WebOps-Club/personal-website-template,2.0,1.0
9,simple-gallery_responsive-template,UNRESOLVED,NaN,0.0,0.0


In [53]:
print(
    multiple_site_decisions[
        "repository_status"
    ].value_counts(
        dropna=False
    )
)

repository_status
UNRESOLVED                     12
PROVISIONAL_MODERATE            2
AMBIGUOUS_MULTIPLE_MODERATE     2
CONFIRMED_STRONG                1
Name: count, dtype: int64


In [54]:
single_validation[
    "validation_status"
] = single_validation.apply(
    classify_repository_validation,
    axis=1
)

In [55]:
def decide_single_repository(
    row
):

    status = row[
        "validation_status"
    ]

    if status == "STRONG_MATCH":

        return pd.Series({
            "repository_status":
                "CONFIRMED_STRONG",

            "selected_repo":
                row[
                    "repo_full_name"
                ]
        })

    if status == "MODERATE_MATCH":

        return pd.Series({
            "repository_status":
                "PROVISIONAL_MODERATE",

            "selected_repo":
                row[
                    "repo_full_name"
                ]
        })

    return pd.Series({
        "repository_status":
            "UNRESOLVED",

        "selected_repo":
            None
    })

In [56]:
single_site_decisions = (
    single_validation
    .apply(
        decide_single_repository,
        axis=1
    )
)

single_site_decisions.insert(
    0,
    "site",
    single_validation[
        "site"
    ].values
)

display(
    single_site_decisions
)

,site,repository_status,selected_repo
0,academic-website-template,UNRESOLVED,NaN
1,conference-website-template,PROVISIONAL_MODERATE,mikepierce/conference-website-template
2,jekyll-inclusion,UNRESOLVED,NaN
3,marketing-agency_one-page-template,UNRESOLVED,NaN
4,material-dashboard,UNRESOLVED,NaN
5,portfolio_one-page-template,UNRESOLVED,NaN
6,react-saas-template,PROVISIONAL_MODERATE,dunky11/react-saas-template
7,simple-product_responsive-template,UNRESOLVED,NaN
8,startbootstrap-3-col-portfolio,CONFIRMED_STRONG,StartBootstrap/startbootstrap-3-col-portfolio
9,startbootstrap-simple-sidebar,PROVISIONAL_MODERATE,StartBootstrap/startbootstrap-simple-sidebar


In [57]:
no_exact_sites = (
    recovery_review.loc[
        recovery_review[
            "review_status"
        ]
        ==
        "UNRESOLVED",
        "site"
    ]
    .tolist()
)

no_exact_decisions = pd.DataFrame({
    "site":
        no_exact_sites,

    "repository_status":
        "UNRESOLVED",

    "selected_repo":
        None
})

no_exact_decisions

,site,repository_status,selected_repo
0,Business-Website,UNRESOLVED,None
1,university-template,UNRESOLVED,None


In [58]:
single_final = single_site_decisions[
    [
        "site",
        "repository_status",
        "selected_repo"
    ]
].copy()

multiple_final = multiple_site_decisions[
    [
        "site",
        "repository_status",
        "selected_repo"
    ]
].copy()

unresolved_final = no_exact_decisions[
    [
        "site",
        "repository_status",
        "selected_repo"
    ]
].copy()

In [59]:
catching_repository_final = pd.concat(
    [
        single_final,
        multiple_final,
        unresolved_final
    ],
    ignore_index=True
)

catching_repository_final = (
    catching_repository_final
    .sort_values("site")
    .reset_index(drop=True)
)

print(
    catching_repository_final.shape
)

display(
    catching_repository_final
)

(30, 3)


,site,repository_status,selected_repo
0,Business-Website,UNRESOLVED,None
1,Discord-bot-website-template,PROVISIONAL_MODERATE,Neovenchy/Discord-bot-website-template
2,Electronics-eCommerce-Shop-With-Admin-Dashboard-NextJS-NodeJS-main,UNRESOLVED,NaN
3,academic-website-template,UNRESOLVED,NaN
4,academicpages.github.io,UNRESOLVED,NaN
5,airspace-bootstrap,UNRESOLVED,NaN
6,beginner-html-site-styled,AMBIGUOUS_MULTIPLE_MODERATE,NaN
7,charity_responsive-template,UNRESOLVED,NaN
8,conference-website-template,PROVISIONAL_MODERATE,mikepierce/conference-website-template
9,coreui-free-react-admin-template,AMBIGUOUS_MULTIPLE_MODERATE,NaN


In [60]:
assert (
    catching_repository_final[
        "site"
    ].nunique()
    ==
    30
)

assert (
    catching_repository_final[
        "site"
    ].duplicated()
    .sum()
    ==
    0
)

print(
    "30 sitios únicos ✅"
)

30 sitios únicos ✅


In [61]:
print(
    catching_repository_final[
        "repository_status"
    ].value_counts(
        dropna=False
    )
)

repository_status
UNRESOLVED                     21
PROVISIONAL_MODERATE            5
AMBIGUOUS_MULTIPLE_MODERATE     2
CONFIRMED_STRONG                2
Name: count, dtype: int64


In [62]:
multiple_validation.to_csv(
    REPORTS_DIR
    / "catching_multiple_candidate_validation.csv",
    index=False
)

multiple_site_decisions.to_csv(
    REPORTS_DIR
    / "catching_multiple_candidate_decisions.csv",
    index=False
)

catching_repository_final.to_csv(
    REPORTS_DIR
    / "catching_repository_final.csv",
    index=False
)

In [63]:
for filename in [
    "catching_single_candidate_validation.csv",
    "catching_multiple_candidate_validation.csv",
    "catching_multiple_candidate_decisions.csv",
    "catching_repository_final.csv"
]:

    path = (
        REPORTS_DIR
        / filename
    )

    print(
        filename,
        "✅"
        if path.exists()
        else "❌"
    )

catching_single_candidate_validation.csv ✅
catching_multiple_candidate_validation.csv ✅
catching_multiple_candidate_decisions.csv ✅
catching_repository_final.csv ✅
